# Celestack Workflow

This is a development workflow for the `celestack` project.

Before release, it will be fully substituted by a CLI, TUI, or GUI.

## Project-level Settings:

In [ ]:
from pathlib import Path

from celestack.stack import FrameStack

PROJECT = "test"

## Initialize the Project Stack

In [ ]:
stack = FrameStack(PROJECT)
stack

## Load the Frames

In [ ]:
stack = FrameStack.from_state(PROJECT)
lf_paths = list(Path("/home/martin/Desktop/tenerife/lf").glob("*.tif"))
df_paths = list(Path("/home/martin/Desktop/tenerife/df").glob("*.tif"))

stack.load_frames(
    lf_paths=lf_paths,
    df_paths=df_paths,
)

stack = FrameStack.from_state(PROJECT)

## Apply the Dark Frames Correction

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.apply_dark_frames_correction()

stack = FrameStack.from_state(PROJECT)
stack.master_dark

## Create the Average Light Frame

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.create_average_light_frame()

stack = FrameStack.from_state(PROJECT)
stack.avg_light

## Create and Apply the Foreground Mask

Start with clustering the pixels of the average light frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)
if stack.avg_light is None:
    msg = "No average light frame found."
    raise ValueError(msg)

alf = stack.avg_light
alf.cluster_pixels(n_clusters=2)
fig = alf.plot_clusters()
fig.show()

Now, turn the clustered pixels into a `Mask` object and add it to the stack.

In [ ]:
alf.initialize_mask(foreground_cluster_labels=[0])

# Explicitly mask/unmask some areas of the image
alf.set_mask_in_box(value=False, y2=1760)  # top part of the image is sky
alf.set_mask_in_box(value=True, y1=2035)  # bottom part of the image is foreground

mask = alf.create_mask()

stack.add_mask(mask)

And finally, add the mask to the stack.

In [ ]:
# Show that the mask has been added to the stack
stack = FrameStack.from_state(PROJECT)
if stack.mask is None or stack.light_frames["P3290058"].mask is None:
    msg = "No mask found in stack or light frame."
    raise ValueError(msg)

stack.mask.plot().show()

## Sky Segmentation

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.segment_sky()

stack = FrameStack.from_state(PROJECT)
if not stack.segment_boxes:
    msg = "No segment boxes found in stack."
    raise ValueError(msg)

stack = FrameStack.from_state(PROJECT)
stack.plot().show()

## Detect Stars in Reference Frame

First, set the reference frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.set_reference_frame("P3290100")

Now, detect stars in the reference frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)
stack.detect_stars_in_ref_frame(n=1000)

Let's plot the stars.

In [ ]:
stack = FrameStack.from_state(PROJECT)

if stack.stars_table is None:
    raise ValueError

fig = stack.plot()
fig.show()

stack.stars_table.df


## Propagate the stars across the stack

Let's test the algo for propagating a single star across the stack.

In [ ]:
import polars as pl

stack = FrameStack.from_state(PROJECT)

st = stack.stars_table
if st is None:
    raise ValueError

st.df.filter(pl.col("frame") == "P3290100")


## Development

The problematic stars:

* 334, 876: The brightest stars, not propagated at all
* 833: One of the stars in the low-right corner which goes loopy

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.plot()